# Physical validation of Fresnel propagation

This notebook compares Fiatlux propagation with Gaussian-beam theory, checks the far-field limit, and demonstrates exact sampled forward/backward recovery.

In [ ]:
import matplotlib.pyplot as plt
import torch

from fiatlux import FFTPropagator, Field, Grid, Spectrum
from fiatlux.core.spectrum import Band

torch.set_default_dtype(torch.float64)

wavelength = 632.8e-9
waist = 0.15e-3
grid = Grid(256, 256, 15e-6, 15e-6, dtype=torch.float64)
x, y = grid.meshgrid()
r2 = x.square() + y.square()
amplitude = torch.exp(-r2 / waist**2).to(torch.complex128)
spectrum = Spectrum(
    magnitude=0, band=Band(wavelength, 0.0, 1.0), samples=1, dtype=torch.float64
)
field = Field(amplitude.unsqueeze(0), grid, spectrum)
rayleigh_distance = torch.pi * waist**2 / wavelength
print(f"Rayleigh distance: {float(rayleigh_distance):.4f} m")

## Gaussian-beam spreading

For an amplitude proportional to \(e^{-r^2/w_0^2}\), theory predicts \(w(z)=w_0\sqrt{1+(z/z_R)^2}\).

In [ ]:
distances = [0.04, 0.08, 0.16, 0.32]
measured = []
expected = []
propagated_fields = []

for distance in distances:
    propagated = FFTPropagator(
        propagation="fresnel", distance=distance
    ).apply(field)
    intensity = propagated.intensity()[0]
    x_out, y_out = propagated.grid.meshgrid()
    measured_waist = torch.sqrt(
        2 * ((x_out.square() + y_out.square()) * intensity).sum()
        / intensity.sum()
    )
    expected_waist = waist * torch.sqrt(
        torch.tensor(1 + (distance / rayleigh_distance) ** 2)
    )
    torch.testing.assert_close(measured_waist, expected_waist, rtol=3e-3, atol=0)
    measured.append(float(measured_waist * 1e3))
    expected.append(float(expected_waist * 1e3))
    propagated_fields.append(propagated)

plt.figure(figsize=(6, 4))
plt.plot(distances, expected, "k-", label="Gaussian theory")
plt.plot(distances, measured, "o", label="Fiatlux FFT Fresnel")
plt.xlabel("Distance z [m]")
plt.ylabel("Beam radius w(z) [mm]")
plt.grid(True)
plt.legend()
plt.show()

## Intensity and phase

The beam spreads while its phase acquires the expected quadratic curvature.

In [ ]:
fig, axes = plt.subplots(2, len(distances), figsize=(14, 6), constrained_layout=True)
for column, (distance, propagated) in enumerate(zip(distances, propagated_fields)):
    extent = [
        float(propagated.grid.x.min() * 1e3), float(propagated.grid.x.max() * 1e3),
        float(propagated.grid.y.min() * 1e3), float(propagated.grid.y.max() * 1e3),
    ]
    axes[0, column].imshow(
        propagated.intensity()[0].cpu(), origin="lower", extent=extent
    )
    axes[1, column].imshow(
        propagated.phase()[0].cpu(), origin="lower", extent=extent,
        cmap="twilight", vmin=-torch.pi, vmax=torch.pi,
    )
    axes[0, column].set_title(f"z = {distance:.2f} m")
    axes[0, column].set_ylabel("Intensity")
    axes[1, column].set_ylabel("Phase [rad]")
    axes[1, column].set_xlabel("x [mm]")
plt.show()

## Fraunhofer limit and reversibility

At long distance, normalized Fresnel intensity converges to Fraunhofer. A sampled propagation by \(+z\) followed by \(-z\) returns the original complex field.

In [ ]:
far_distance = 10.0
fresnel_far = FFTPropagator(
    propagation="fresnel", distance=far_distance
).apply(field)
fraunhofer = FFTPropagator(focal_length=far_distance).apply(field)
fi = fresnel_far.intensity()[0] / fresnel_far.intensity()[0].max()
ff = fraunhofer.intensity()[0] / fraunhofer.intensity()[0].max()
relative_error = torch.linalg.vector_norm(fi - ff) / torch.linalg.vector_norm(ff)
print(f"Far-field normalized-intensity error: {float(relative_error):.3e}")

forward = FFTPropagator(propagation="fresnel", distance=0.25).apply(field)
recovered = FFTPropagator(propagation="fresnel", distance=-0.25).apply(forward)
torch.testing.assert_close(
    recovered.complex_amplitude, field.complex_amplitude, rtol=2e-12, atol=2e-12
)
recovery_error = torch.linalg.vector_norm(
    recovered.complex_amplitude - field.complex_amplitude
) / torch.linalg.vector_norm(field.complex_amplitude)
print(f"Forward/backward complex-field error: {float(recovery_error):.3e}")